### Importing the packages

In [ ]:
import pandas as pd
from itertools import chain
from collections import Counter
from os.path import join

### Paths and parameters

#### Pipeline input folders

In [ ]:
pangenome = join("03-pangenomes", "all")
eggnog_mapper = join("06-pangenome-annotation", "mapper")
classification = '02-GTDB'
COG_cats = join("utils", "COG_cats.tsv")

#### Pipeline output folders

In [ ]:
task_root = "06-pangenome-annotation"
processed_output = join(task_root, "processed_output")

!mkdir -p $task_root $processed_output

#### Tool pointers and parameters

In [ ]:
pa_file = join(pangenome, 'matrix.csv')
mapper_file = join(eggnog_mapper, 'all.emapper.annotations')
group_assignments = join(classification, 'filtered_classification_table')

### Importing the data

In [ ]:
# COG category index
cog_cats = pd.read_table(COG_cats, sep='\t', header = None)
cog_cats = dict(zip(cog_cats[0], cog_cats[1]))

In [ ]:
# group assigments
groups = pd.read_table(group_assignments, sep = "\t", header = None, names = ['accession', 'group'])

In [ ]:
# P/A matrix
pa = pd.read_table(pa_file, sep = ",", low_memory = False).set_index('Gene')
pa = pa.iloc[:, [2] + list(range(13, pa.shape[1]))]

In [ ]:
# eggNOG mappings
egg_maps = pd.read_table(mapper_file, sep = "\t", usecols = [0,6], skiprows = 5, skipfooter = 3, header = None, names = ['Gene', 'COG'])
egg_maps = egg_maps.set_index('Gene')

### Setting the partitioning thresholds

In [ ]:
ca = 90
au = 100/(pa.shape[1]-1)

### Extracting subpangenomes

In [ ]:
def extract_subpangenome(pa, assignments, group_label):
    accessions = assignments[assignments["group"] == group_label]['accession'].to_list()
    subpa = pa[accessions]
    subpa['No. isolates'] = subpa.notna().sum(axis = 1)
    return subpa

In [ ]:
subpans = {class_label : extract_subpangenome(pa, groups, class_label) for class_label in groups['group'].unique()}
subpans['all'] = pa

### Partitioning the pangenomes

In [ ]:
def partition_pangenome(pa, ca_threshold = 90, au_threshold = 15, print_size = False):
    partitions = {}
    
    pa['presence_ratio'] = pa['No. isolates'] / (pa.shape[1]-1) * 100

    partitions['core'] = pa[pa["presence_ratio"] >= ca_threshold]
    partitions['accessory'] = pa[(pa["presence_ratio"] > au_threshold) & (pa["presence_ratio"] < ca_threshold)]
    partitions['unique'] = pa[(pa["presence_ratio"] <= au_threshold) & (pa["presence_ratio"] > 0)]

    if print_size:
        print("Core:\t" + str(partitions['core'].shape))
        print("Accessory:\t" + str(partitions['acc'].shape))
        print("Unique:\t" + str(partitions['unique'].shape))

    return partitions

In [ ]:
subpans_partitioned = {group: partition_pangenome(pa, ca, 100/(pa.shape[1]-1)) for group, pa in subpans.items()}

### Getting the size of the partitioned pangenomes

In [ ]:
partition_sizes_groups = {}
for group, pan in subpans_partitioned.items():
    partition_sizes_this_group = {}
    for partition, pa in pan.items():
        partition_sizes_this_group[partition] = subpans_partitioned[group][partition].shape[0]
    partition_sizes_groups[group] = partition_sizes_this_group
partition_sizes_groups = pd.DataFrame(partition_sizes_groups)

In [ ]:
partition_sizes_groups

In [ ]:
partition_sizes_groups.to_csv(join(task_root, 'partition_sizes'), sep = '\t')

### Map to COG annotations

In [ ]:
def map_eggnog(pa, eggnog):
    return pd.merge(pa, eggnog, how = "left", left_index = True, right_index = True)['COG'].fillna('-').to_list()

In [ ]:
subpans_partitioned_cog = {}
for group, partitioned_subpa in subpans_partitioned.items():
    this_subpan_cog = {}
    for partition, pa_matrix in partitioned_subpa.items():
        this_subpan_cog[partition] = map_eggnog(pa_matrix, egg_maps)
    subpans_partitioned_cog[group] = this_subpan_cog

### Count COG annotations

In [ ]:
def count_eggnog(anno_list, relative = True):
    counts = dict(Counter(list(chain(*[list(i) for i in anno_list]))))
    counts = pd.DataFrame.from_dict([counts]).T.squeeze().sort_index()
    if relative:
        counts = counts / counts.sum() * 100
    return counts.to_dict()

In [ ]:
subpans_partitioned_cog_counts = {}
for group, partitioned_subpa in subpans_partitioned_cog.items():
    this_subpan_cog_counts = {}
    for partition, cogs in partitioned_subpa.items():
        this_subpan_cog_counts[partition] = count_eggnog(cogs)
    subpans_partitioned_cog_counts[group] = this_subpan_cog_counts

### Summarise count dataframes

In [ ]:
def summarise_counts(counts, n_dec = 3):
    return pd.DataFrame(counts).fillna(0).round(n_dec).sort_index()

In [ ]:
count_summaries={}
for group, partition_counts in subpans_partitioned_cog_counts.items():
    count_summaries[group] = summarise_counts(partition_counts)
count_summaries = pd.concat(count_summaries.values(), keys = count_summaries.keys(), names = ['group', 'category'])

In [ ]:
count_summaries

### Saving results

In [ ]:
count_summaries_melt = count_summaries.melt(ignore_index = False, var_name = 'partition', value_name = 'fraction')
count_summaries_melt.to_csv(join(processed_output, 'summary'), sep = "\t")